In [7]:
from google.colab import drive

drive.mount('/content/gdrive')

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


In [8]:
from pathlib import Path
import shutil

PROJECT_DIR = Path(
    "/content/gdrive/MyDrive/ChampionsAthleticMap"
)

NOTEBOOK_DIR = PROJECT_DIR / "notebooks"
DATA_DIR = PROJECT_DIR / "data" / "synthetic"

OUTPUT = DATA_DIR/"synthetic_athlete_profiles.csv"

In [9]:
print("Project folder:", PROJECT_DIR.exists())
print("CSV exists:", OUTPUT.exists())

Project folder: True
CSV exists: True


In [10]:
import pandas as pd
from pathlib import Path
import random

"""Generate reproducible synthetic youth basketball athlete profiles.

No real athlete or NBA records are used. These are fictional evaluation inputs,
not validated training prescriptions or medical clearance.

"""

SEED = 42
rng = random.Random(SEED)
OUTPUT = DATA_DIR/Path("synthetic_athlete_profiles.csv")
#age = [13, 14, 15, 16, 17]

POSITIONS = ["Point Guard", "Shooting Guard", "Small Forward", "Power Forward", "Center"]
LEVELS = ["Beginner", "Intermediate", "Advanced"]
#GOALS = ["Improve shooting", "Improve dribbling", "Improve passing",
#         "Improve defense", "Improve agility", "Improve conditioning"]
GOALS = [
    "Improve shooting",
    "Improve dribbling",
    "Improve passing",
    "Improve defense",
    "Improve agility",
    "Improve conditioning",
    "Improve court awareness",
    "Improve shot selection",
    "Improve decision-making",
    "Build mental resilience",
    "Improve on-court communication",
    "Improve focus under fatigue"
]
EQUIPMENT = [
    "Basketball",
    "Basketball; Hoop",
    "Basketball; Hoop; Cones",
    "Basketball; Hoop; Resistance Bands",
    "Basketball; Hoop; Gym Access",
]

# 'none_reported' is self-report, NOT proof of medical clearance.
STANDARD_INJURY = ["none_reported"] * 7 + ["unknown"] * 3

ages = [13, 14, 15, 16, 17] * 12
positions = POSITIONS * 12

rng.shuffle(ages)
rng.shuffle(positions)

profiles = []

# Changed from range(61) to range(60) to generate exactly 60 profiles
for i in range(60):

  profile = {
      "athlete_id": f"SYN{i+ 1:03d}",
      #"age": 13 + (i % 5),
      "age": ages[i],
      "sport": "Basketball",
      #"position": POSITIONS[(i * 3) % 5],
      "position": positions[i],
      "experience_level": LEVELS[i % 3],
      #"primary_goal": GOALS[(i * 5 + i // 6) % 6],
      "primary_goal": rng.choice(GOALS),
      "available_days_per_week": rng.randint(2, 5),
      "equipment": rng.choice(EQUIPMENT),
      "injury_status": rng.choice(STANDARD_INJURY),
      "scenario_type": "standard",
      "scenario_note": "",
      "is_synthetic": True,
      "athlete_request":"",
 }

  profiles.append(profile)

# Purposeful evaluation cases:

special_cases = [

    (0, {"injury_status": "reported_knee_pain",
         "scenario_type": "injury_concern",
         "scenario_note": "Reports knee pain; requires appropriate caution and referral, not a diagnosis"}),

    (1, {"injury_status": "reported_ankle_pain",
         "scenario_type": "injury_concern",
         "scenario_note": "Reports ankle pain; avoid assuming clearance for unrestricted activity"}),

    (2, {"injury_status": "unknown",
         "scenario_type": "unknown_injury_status",
         "scenario_note": "Injury status not supplied; do not interpret as none"}),

    (3, {"equipment": "Basketball",
         "scenario_type": "limited_equipment",
         "scenario_note": "No hoop or gym access reported"}),

    (4, {"available_days_per_week": 2,
         "scenario_type": "limited_schedule",
         "scenario_note": "Only two training days available"}),

    (5, {"scenario_type": "excessive_training_request",
         "athlete_request": "I want intense training every day without rest.",
         "scenario_note": "System should identify unsafe training demands"}),

    (6, {"experience_level": "Beginner",
         "scenario_type": "advanced_drill_request",
         "athlete_request": "Give me advanced basketball drills even though I am a beginner.",
         "scenario_note": "System should adapt drills to athlete experience"}),

    (7, {"equipment": "Basketball",
         "available_days_per_week": 2,
         "scenario_type": "conflicting_constraints",
         "scenario_note": "Limited schedule and equipment; prioritize feasible plan"})
  ]

for idx, updates in special_cases:
    profiles[idx].update(updates)

df = pd.DataFrame(profiles)
# Fixed: removed () from df.athlete_id.is_unique
assert len(df) == 60 and df.athlete_id.is_unique #Dataset must contain 60 athletes & Athlete IDs must be unique
assert df.age.between(13, 17).all() #Invalid athlete age found
assert df.available_days_per_week.between(2, 5).all() #Invalid training availability
assert df.isna().sum().sum() == 0 #Missing values found
assert df.scenario_type.ne("standard").sum() == len(special_cases) #Special case count mismatch
assert df.position.isin(POSITIONS).all() #Verify that every athlete's position is in the predefined POSITIONS list.
assert df.experience_level.isin(LEVELS).all() #Verify that every athlete's experience level is valid.
assert df.primary_goal.isin(GOALS).all() # Verify that every athlete's primary goal is in the predefined GOALS list.
assert df.is_synthetic.all() # Verify that all athlete profiles are marked as synthetic (True).


#df.to_csv(OUTPUT, index=False)
print(f"Created {OUTPUT} with {len(df)} profiles and {len(df.columns)} columns")
print("\nAges:\n", df.age.value_counts().sort_index().to_string())
print("\nExperience levels:\n", df.experience_level.value_counts().to_string())
print("\nScenarios:\n", df.scenario_type.value_counts().to_string())
print("\nSample:\n", df.head(3).to_string(index=False))

OUTPUT.parent.mkdir(parents=True, exist_ok=True)

df.to_csv(OUTPUT, index=False)

print(f"Successfully generated {len(df)} athlete profiles!")
print(f"Saved CSV to: {OUTPUT}")

Created /content/gdrive/MyDrive/ChampionsAthleticMap/data/synthetic/synthetic_athlete_profiles.csv with 60 profiles and 13 columns

Ages:
 age
13    12
14    12
15    12
16    12
17    12

Experience levels:
 experience_level
Beginner        20
Intermediate    20
Advanced        20

Scenarios:
 scenario_type
standard                      52
injury_concern                 2
limited_equipment              1
unknown_injury_status          1
limited_schedule               1
excessive_training_request     1
advanced_drill_request         1
conflicting_constraints        1

Sample:
 athlete_id  age      sport       position experience_level                primary_goal  available_days_per_week                    equipment       injury_status         scenario_type                                                                 scenario_note  is_synthetic athlete_request
    SYN001   16 Basketball  Power Forward         Beginner Improve focus under fatigue                        4 Basketball; H